In [1]:
import os
import csv
import pycdlib
from datetime import datetime

In [2]:
SOURCE_DIR = r"G:\thecall\test"
DEST_DIR   = r"G:\c_out"
CSV_PATH   = os.path.join(DEST_DIR, "inventory.csv")
ERROR_LOG  = os.path.join(DEST_DIR, "errors.log")

In [3]:
# ---------- Utilities ----------

def safe_path(path):
    """Enable Windows long paths."""
    path = os.path.abspath(path)
    if not path.startswith("\\\\?\\"):
        return "\\\\?\\" + path
    return path


def clean_name(name):
    """Remove illegal Windows filename characters."""
    return "".join(c if c not in '<>:"\\|?*' else '_' for c in name)


def get_iso_datetime(iso, rr_path):
    """Try to extract modified datetime from ISO metadata."""
    try:
        rec = iso.get_record(rr_path=rr_path)
        dt = rec.date  # pycdlib returns a datetime-like object
        if dt:
            return dt.strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        pass
    return ""


def log_error(message):
    with open(ERROR_LOG, "a", encoding="utf-8") as f:
        f.write(message + "\n")

In [4]:
# ---------- Core extraction ----------

def extract_iso(iso_path, output_dir, writer):
    iso = pycdlib.PyCdlib()
    iso.open(iso_path)

    iso_name = os.path.basename(iso_path)

    for path, dirs, files in iso.walk(rr_path='/'):
        for file in files:
            try:
                rr_file_path = os.path.join(path, file)

                # Clean names for Windows safety
                clean_file = clean_name(file)
                rel_dir = path.lstrip('/')
                rel_dir = os.path.normpath(rel_dir)

                out_dir = os.path.join(output_dir, rel_dir)
                out_path = os.path.join(out_dir, clean_file)

                os.makedirs(safe_path(out_dir), exist_ok=True)

                # Extract file
                iso.get_file_from_iso(
                    rr_path=rr_file_path,
                    local_path=safe_path(out_path)
                )

                # Metadata
                mod_time = get_iso_datetime(iso, rr_file_path)

                # Write to CSV
                writer.writerow({
                    "iso_file": iso_name,
                    "file_name": clean_file,
                    "folder": rel_dir,
                    "full_path": out_path,
                    "modified_time": mod_time
                })

            except Exception as e:
                log_error(f"{iso_name} | {rr_file_path} | {str(e)}")

    iso.close()




In [5]:
# ---------- Main pipeline ----------

def main():
    os.makedirs(DEST_DIR, exist_ok=True)

    with open(CSV_PATH, "w", newline="", encoding="utf-8") as csvfile:
        fieldnames = ["iso_file", "file_name", "folder", "full_path", "modified_time"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for root, _, files in os.walk(SOURCE_DIR):
            for file in files:
                if file.lower().endswith(".iso"):
                    iso_path = os.path.join(root, file)
                    out_folder = os.path.join(DEST_DIR, os.path.splitext(file)[0])

                    print(f"Processing: {iso_path}")
                    os.makedirs(out_folder, exist_ok=True)

                    extract_iso(iso_path, out_folder, writer)

    print("Done.")



In [6]:

if __name__ == "__main__":
    main()

Processing: G:\thecall\test\2\CALL 04-18-04-25.ISO


PyCdlibInvalidInput: Cannot fetch a rr_path from a non-Rock Ridge ISO